# Week 8 - Notebook 1: Setup RAG System

## Goal
Set up the ChromaDB vector database with 800k products for RAG:
1. Load dataset from HuggingFace Hub
2. Create embeddings using SentenceTransformer
3. Store in ChromaDB
4. Test similarity search

## Time: 20-30 minutes (embedding takes time)

In [ ]:
import sys
sys.path.append('..')

import os
from dotenv import load_dotenv
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

from src.utils.items import Item
from src.config import config

# Load environment
load_dotenv()

print("✅ Environment loaded")

## Configuration

In [ ]:
config.display()

## Step 1: Load Dataset

In [ ]:
print(f"Loading dataset: {config.DATASET_NAME}")
train, val, test = Item.from_hub(config.DATASET_NAME)
all_items = train + val + test

print(f"\n✅ Loaded:")
print(f"   Training: {len(train):,} items")
print(f"   Validation: {len(val):,} items")
print(f"   Test: {len(test):,} items")
print(f"   Total: {len(all_items):,} items")

## Step 2: Initialize ChromaDB

In [ ]:
# Create ChromaDB client
chroma_client = chromadb.PersistentClient(path="../data/chroma")

# Create or get collection
collection_name = "products"

try:
    # Try to delete existing collection
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection: {collection_name}")
except:
    print(f"No existing collection to delete")

# Create new collection
collection = chroma_client.create_collection(name=collection_name)
print(f"✅ Created ChromaDB collection: {collection_name}")

## Step 3: Load Embedding Model

In [ ]:
# Load SentenceTransformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ Loaded embedding model: all-MiniLM-L6-v2")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

## Step 4: Create Embeddings and Store

This will take 20-30 minutes for 800k items

In [ ]:
# Process in batches
BATCH_SIZE = 1000
total_batches = (len(all_items) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Processing {len(all_items):,} items in {total_batches} batches...")
print("This will take 20-30 minutes...\n")

for i in tqdm(range(0, len(all_items), BATCH_SIZE), desc="Embedding batches"):
    batch = all_items[i:i+BATCH_SIZE]
    
    # Prepare data
    ids = [str(item.id) for item in batch]
    documents = [item.prompt or item.summary or item.title for item in batch]
    metadatas = [
        {
            "title": item.title,
            "category": item.category,
            "price": item.price,
        }
        for item in batch
    ]
    
    # Create embeddings
    embeddings = model.encode(documents).astype(float).tolist()
    
    # Add to ChromaDB
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings,
    )

print(f"\n✅ Added {len(all_items):,} items to ChromaDB")
print(f"   Collection count: {collection.count():,}")

## Step 5: Test Similarity Search

In [ ]:
# Test query
test_query = "wireless bluetooth headphones with noise cancellation"

print(f"Test query: {test_query}\n")

# Create embedding for query
query_embedding = model.encode([test_query]).astype(float).tolist()

# Search
results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

print("Top 5 similar products:\n")
for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
    print(f"{i}. {metadata['title']}")
    print(f"   Category: {metadata['category']}")
    print(f"   Price: ${metadata['price']:.2f}")
    print(f"   Description: {doc[:100]}...\n")

## Summary

✅ RAG system setup complete!

**What we did:**
1. Loaded 800k products from HuggingFace Hub
2. Created embeddings using SentenceTransformer
3. Stored in ChromaDB vector database
4. Tested similarity search

**ChromaDB location:** `../data/chroma/`

**Next step:** `02_test_agents.ipynb` - Test individual agents